# Mouse oral swabs PCA and mean Z-scores

Figure 1b and Supplementary Figure 2a. The PCA decision boundary is descriptive. The heatmap ranks families by absolute group differences in mean Z-scores.

In [ ]:
from pathlib import Path

# Jupyter normally starts the kernel in the notebook directory.
here = Path.cwd()
repo_root = next(parent for parent in (here, *here.parents)
                 if (parent / "requirements.txt").is_file()
                 and (parent / "mouse").is_dir() and (parent / "human").is_dir())
data_dir = repo_root / "data"
output_dir = repo_root / "outputs" / 'mouse/02_pca_and_z_scores'
output_dir.mkdir(parents=True, exist_ok=True)
notebook_output = output_dir


## Load the abundance workbook and identify metadata

In [ ]:
import pandas as pd

# Define the file path
path = data_dir / 'Mouse_Data_For_PCA.xlsx'

# Read the Excel file into a DataFrame
df = pd.read_excel(path)
# Remove rows where all values are NaN
df = df.dropna(how='all')

df


In [ ]:
# Define label columns (metadata)
label_cols = ['Full ID', 'Group', 'Timepoint', 'Sex']

# Define feature columns (everything else)
feature_cols = [col for col in df.columns if col not in label_cols]

# Create separate DataFrames
df_labels = df[label_cols]
df_features = df[feature_cols]


## Principal component analysis

In [ ]:
# ============================================================
# PCA Visualization with Linear Boundary & Accuracy Annotation
# Save Top 10 Differential Features
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# -------------------------------
# Global style (font + scaling)
# -------------------------------
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,            # Base font size
    'axes.titlesize': 18,       # Title size
    'axes.labelsize': 16,       # Axis label size
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'legend.title_fontsize': 14,
    'figure.titlesize': 20
})

plt.style.use("default")

plt.rcParams['pdf.fonttype'] = 42       # Use TrueType fonts (not Type 3)
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'   # Keep text editable in SVGs

# -------------------------------
# Ensure group order
# -------------------------------
df_labels["Group"] = pd.Categorical(
    df_labels["Group"],
    categories=["B6", "M83(ho)"],
    ordered=True
)

# -------------------------------
# Compute global Z-scores
# -------------------------------
scaler = StandardScaler()
df_z = pd.DataFrame(
    scaler.fit_transform(df_features),
    columns=df_features.columns,
    index=df_features.index
)

# -------------------------------
# Compute mean Z-score per group
# -------------------------------
df_mean_z = df_z.groupby(df_labels["Group"]).mean().T
df_mean_z["Difference"] = df_mean_z["B6"] - df_mean_z["M83(ho)"]

# -------------------------------
# Filter features by absolute difference threshold
# -------------------------------
threshold = 0.4
selected_features = df_mean_z.loc[df_mean_z["Difference"].abs() > threshold].index

num_selected = len(selected_features)
num_total = df_features.shape[1]
print(f"Kept {num_selected} of {num_total} taxa ({num_selected / num_total:.1%}) with |ΔZ| > {threshold}")

# -------------------------------
# Identify Top 10 absolute differences
# -------------------------------
top10_features = (
    df_mean_z.reindex(selected_features)
    .loc[:, "Difference"]
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .index
)

top10_features = list(top10_features)
print("\nTop 10 most discriminative features (by |ΔZ|):")
for i, f in enumerate(top10_features, 1):
    print(f"{i}. {f}")

# -------------------------------
# PCA on filtered features
# -------------------------------
df_features_filtered = df_features[selected_features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features_filtered)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Get variance explained by each PC
var1, var2 = pca.explained_variance_ratio_ * 100

# -------------------------------
# Logistic regression on PC space
# -------------------------------
y_binary = (df_labels["Group"] == "M83(ho)").astype(int)
model = LogisticRegression(random_state=42)
acc = cross_val_score(model, X_pca, y_binary, cv=5, scoring='accuracy').mean()
print(f"\nMean 5-fold logistic regression accuracy on PC1–PC2: {acc*100:.1f}%")

# Fit once on full data (for plotting)
model.fit(X_pca, y_binary)

# -------------------------------
# Build plotting DataFrame
# -------------------------------
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Group'] = df_labels['Group'].values
pca_df['Sex'] = df_labels['Sex'].values

# -------------------------------
# Styling setup
# -------------------------------
palette = {"B6": "#42A5F5", "M83(ho)": "#EF5350"}
plot_params = dict(s=110, linewidth=1.4, alpha=0.9)

fig, ax = plt.subplots(figsize=(9, 7))

# -------------------------------
# Plot each Group × Sex combo
# -------------------------------
for grp, color in palette.items():
    subset = pca_df[pca_df["Group"] == grp]

    # Male points → no edge outline
    male_sub = subset[subset["Sex"].str.lower().isin(["male", "m"])]
    sns.scatterplot(
        data=male_sub, x='PC1', y='PC2',
        color=color, edgecolor=None,
        label=f"{grp} - Male", ax=ax, **plot_params
    )

    # Female points → black outline
    female_sub = subset[subset["Sex"].str.lower().isin(["female", "f"])]
    sns.scatterplot(
        data=female_sub, x='PC1', y='PC2',
        color=color, edgecolor='black',
        label=f"{grp} - Female", ax=ax, **plot_params
    )

# -------------------------------
# Compute and plot decision boundary
# -------------------------------
x_min, x_max = pca_df["PC1"].min() - 1, pca_df["PC1"].max() + 1
y_min, y_max = pca_df["PC2"].min() - 1, pca_df["PC2"].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

ax.contour(xx, yy, Z, levels=[0.5], colors='black', linestyles='--', linewidths=1.3)

# -------------------------------
# Labels and title
# -------------------------------
ax.set_title(
    f"PCA of Microbiota Abundance Profiles (|ΔZ| > {threshold})\n"
    f"Families retained: {num_selected}/{num_total}\n"
    f"Linear separability accuracy: {acc*100:.1f}%",
    fontsize=18, pad=20, fontweight='bold'
)
ax.set_xlabel(f"PC1 ({var1:.1f}% variance)", fontsize=16)
ax.set_ylabel(f"PC2 ({var2:.1f}% variance)", fontsize=16)
ax.tick_params(labelsize=13)

# -------------------------------
# Legend
# -------------------------------
ax.legend(
    title="Group / Sex", fontsize=13, title_fontsize=14,
    loc="best", frameon=True, facecolor='white', framealpha=0.9
)

plt.tight_layout()

# output_path = notebook_output
# plt.savefig(output_path / "Mouse_PCA.pdf", format="pdf", bbox_inches="tight")

plt.show()


## Rank and plot group mean Z-scores

In [ ]:
# ============================================================
# Top 10 Features (Taxa) – Bar + Scatter + Significance Stars
# + Store Significant Differences
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import ttest_ind

# --- Prepare data ---
df_plot = df_features[top10_features].copy()
df_plot["Group"] = df_labels["Group"].values

# --- Styling ---

palette = {"B6": "#42A5F5", "M83(ho)": "#EF5350"}

# Helper to turn p-values into stars
def p_to_star(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# Dictionary to store significant taxa results
sig_dict = {}

for feature in top10_features:
    # --- Compute t-test between groups ---
    b6 = df_plot.loc[df_plot["Group"] == "B6", feature]
    m83 = df_plot.loc[df_plot["Group"] == "M83(ho)", feature]
    tstat, pval = ttest_ind(b6, m83, equal_var=False, nan_policy='omit')

    # Compute mean difference (M83 - B6)
    diff = m83.mean() - b6.mean()

    star = p_to_star(pval)

    # --- Store significance results ---
    if pval < 0.05:
        direction = "increased in M83(ho)" if diff > 0 else "decreased in M83(ho)"
        sig_dict[feature] = {
            "p_value": pval,
            "significance": star,
            "direction": direction
        }


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

plt.style.use('default')

# ============================================================
# Mean Z-scores: Top 16 Microbiota Families by |B6–M83(ho) ΔZ|
# ============================================================

# -------------------------------
# Global font and style settings
# -------------------------------
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 20
})
plt.rcParams['pdf.fonttype'] = 42      # TrueType (editable text)
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'  # Editable SVG text

# -------------------------------
# Data preparation
# -------------------------------
# Ensure **Group** order (instead of Treatment)
df_labels["Group"] = pd.Categorical(
    df_labels["Group"],
    categories=["B6", "M83(ho)"],
    ordered=True
)

# Sort samples by Group
sorted_idx = df_labels.sort_values("Group").index
df_labels_sorted = df_labels.loc[sorted_idx]
df_features_sorted = df_features.loc[sorted_idx]

# Compute global Z-scores (relative to dataset mean)
scaler = StandardScaler()
df_z = pd.DataFrame(
    scaler.fit_transform(df_features_sorted),
    columns=df_features_sorted.columns,
    index=df_features_sorted.index
)

# Compute mean Z-score per condition
df_mean_z = df_z.groupby(df_labels_sorted["Group"]).mean().T

# Compute difference between groups
df_mean_z["Difference"] = df_mean_z["B6"] - df_mean_z["M83(ho)"]

# Sort by absolute difference (largest magnitude first)
top_n = 16
abs_sorted_idx = df_mean_z["Difference"].abs().sort_values(ascending=False).head(top_n).index
df_mean_z_top = df_mean_z.loc[abs_sorted_idx]

# Drop the difference column for plotting
df_mean_z_plot = df_mean_z_top[["B6", "M83(ho)"]]

# -------------------------------
# Significance annotation matrix
# -------------------------------
annotations = pd.DataFrame('', index=df_mean_z_plot.index, columns=df_mean_z_plot.columns)
for family, sig_info in sig_dict.items():
    if family in annotations.index:
        annotations.loc[family, "M83(ho)"] = sig_info.get("significance", "")

# -------------------------------
# Plot heatmap (coolwarm)
# -------------------------------
fig, ax = plt.subplots(figsize=(8, 9))

sns.heatmap(
    df_mean_z_plot,
    cmap="coolwarm",
    center=0,
    vmin=-0.5, vmax=0.5,                      # <<< fixed color scale
    cbar_kws={'label': 'Mean Z-score', 'shrink': 0.8},
    yticklabels=True,
    annot=annotations,
    fmt='',
    annot_kws={"fontsize": 13, "color": "black", "va": "center", "ha": "center"},
    ax=ax
)

# Colorbar
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=12)
cbar.set_label("Mean Z-score", fontsize=15, labelpad=10)

# Formatting
ax.set_title("Top 16 Microbiota Families by |B6–M83(ho) ΔZ|", pad=20)
ax.set_ylabel("Bacterial Families", labelpad=10)
ax.set_xlabel("Condition", labelpad=12)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

# -------------------------------
# Save publication‑ready PDF
# -------------------------------
save_path = notebook_output / 'Z_Scores_Mouse.pdf'
plt.savefig(save_path, format="pdf", bbox_inches="tight", dpi=300)
plt.show()

print(f"Figure saved successfully to:\n{save_path}")
